<a href="https://colab.research.google.com/github/thushanch/GEDI/blob/main/2_1_Catchment_Delineation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 1 — Catchment Delineation + DEM Crop
**AGBD to Hydrology Elasticity across Sri Lankan Catchments**

This notebook does exactly Step 1 of the methodology:
1. Delineate the drainage area upstream of each gauge.
2. Crop a Digital Elevation Model to each catchment.
3. Save everything to Google Drive for reuse.

**Approach.** Delineation runs on **MERIT Hydro** (90 m, hydrologically conditioned,
global). The gauge is snapped to the river cell whose upstream area best matches the
published drainage area from the Irrigation Department table. The catchment polygon is
then used to crop the **Copernicus GLO-30** DEM (30 m) for later slope and elevation work.

Run the cells top to bottom. Delineation is roughly one minute per station.

In [27]:
# 1. Install dependencies (once per Colab session)
!pip -q install pysheds geemap geedim rasterio geopandas pyproj shapely --upgrade
print("Dependencies installed. If Colab asks to RESTART, restart then re-run from cell 2.")

Dependencies installed. If Colab asks to RESTART, restart then re-run from cell 2.


In [28]:
# 2. Imports, Earth Engine auth, Drive mount
import os, math, json, warnings
import numpy as np
import ee, geemap
import rasterio
from rasterio.mask import mask as rio_mask
import geopandas as gpd
from shapely.geometry import shape, mapping
from shapely.ops import unary_union
from pyproj import Geod
from pysheds.grid import Grid
from google.colab import drive
warnings.filterwarnings("ignore")

PROJECT = "music-project-466416"        # your GEE project
ee.Authenticate()
ee.Initialize(project=PROJECT)
print("Earth Engine ready ->", ee.String("ok").getInfo())

drive.mount("/content/drive")
OUT_DIR = "/content/drive/MyDrive/AGBD_Hydrology/01_catchments"   # results go here
WORK    = "/content/work"                                          # scratch, cleared each run
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
print("Output folder:", OUT_DIR)

Earth Engine ready -> ok
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Output folder: /content/drive/MyDrive/AGBD_Hydrology/01_catchments


In [29]:
# 3. Station table
# Coordinates transcribed from Station_Coordinates.pdf (Sri Lanka Irrigation Dept,
# Principal Hydrometric Stations 2020/21), WGS84, converted DMS -> decimal degrees.
# pub_area_km2 = published drainage area, used for snapping and validation.
# Only gauges present in Streamflow_2020_to_mid2024.csv are listed.
# NOTE: "Buttala" is in the streamflow CSV but has no coordinate in the PDF, so it is omitted.
#       Add it here as (name, lat, lon, area, basin) if you obtain a coordinate.

STATIONS = [
    # name,            lat,     lon,     pub_area_km2, basin
    ("Norwood",        6.8445,  80.6076,   97, "Kelani Ganga"),
    ("Kithulgala",     6.9906,  80.4122,  383, "Kelani Ganga"),
    ("Deraniyagala",   6.9244,  80.3378,  183, "Kelani Ganga"),
    ("Holombuwa",      7.1853,  80.2647,  155, "Kelani Ganga"),
    ("Glencourse",     6.9744,  80.1828, 1463, "Kelani Ganga"),
    ("Hanwella",       6.9094,  80.0794, 1782, "Kelani Ganga"),
    ("Rathnapura",     6.6783,  80.3942,  603, "Kalu Ganga"),
    ("Ellagawa",       6.7319,  80.2100, 1393, "Kalu Ganga"),
    ("Millakanda",     6.6322,  80.1897,  780, "Kalu Ganga"),
    ("Putupaula",      6.6017,  80.0572, 2598, "Kalu Ganga"),
    ("Urawa",          6.2367,  80.5717,   59, "Nilwala Ganga"),
    ("Pitabeddara",    6.2131,  80.4753,  310, "Nilwala Ganga"),
    ("Wellawaya",      6.7097,  81.1111,  172, "Kirindi Oya"),
    ("Tanamalwila",    6.4683,  81.1342,  749, "Kirindi Oya"),
    ("Kudaoya",        6.5247,  81.1233,  291, "Kirindi Oya"),
    ("Katharagama",    6.4156,  81.3308,  787, "Menik Ganga"),
    ("Nakkala",        6.8950,  81.2969,  216, "Kumbukkan Oya"),
    ("Siyambalanduwa", 6.9050,  81.5433,  295, "Heda Oya"),
    ("Padiyathalawa",  7.3836,  81.1919,  159, "Maduru Oya"),
    ("Thaldena",       7.0908,  81.0481,  276, "Mahaweli Ganga"),
    ("Calidoniya",     6.9019,  80.6978,  148, "Mahaweli Ganga"),
    ("Nawalapitiya",   7.0475,  80.5344,  176, "Mahaweli Ganga"),
    ("Peradeniya",     7.2675,  80.6083, 1168, "Mahaweli Ganga"),
    ("Giriulla",       7.3250,  80.1147, 1191, "Maha Oya"),
    ("Badalgama",      7.3000,  79.9797, 1360, "Maha Oya"),
    ("Dunamale",       7.1156,  80.0806,  153, "Aththanagalu Oya"),
]

# Optional manual snap override for tricky gauges near junctions.
# Put a (lon, lat) here to force the pour point instead of auto-snapping.
SNAP_OVERRIDE = {
    # "Peradeniya": (80.6090, 7.2670),
}
print(len(STATIONS), "stations loaded")

26 stations loaded


In [30]:
# 4. Helper functions
GEOD = Geod(ellps="WGS84")
DIRMAP = (64, 128, 1, 2, 4, 8, 16, 32)   # ESRI / MERIT Hydro D8 convention (pysheds default)

MERIT = ee.Image("MERIT/Hydro/v1_0_1")                 # dir + upa bands, ~90 m
GLO30 = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM").mosaic()  # 30 m DEM

def geodesic_area_km2(geom):
    a, _ = GEOD.geometry_area_perimeter(geom)
    return abs(a) / 1e6

def km_to_deg(km, lat):
    return km / 111.0, km / (111.0 * math.cos(math.radians(lat)))

def dl(img, band, region, scale, path):
    """Download one EE band to a local GeoTIFF (geedim tiling handles large areas)."""
    if os.path.exists(path):
        os.remove(path)
    geemap.download_ee_image(img.select(band), path, region=region, scale=scale, crs="EPSG:4326")
    return path

def largest_polygon(polys):
    u = unary_union(polys)
    if u.geom_type == "MultiPolygon":
        return max(u.geoms, key=lambda p: p.area)
    return u

In [31]:
# 5. Delineation for a single station (with automatic window growth)
def delineate(name, lat, lon, pub_area, basin, max_tries=3):
    half_km = 12 + 1.6 * math.sqrt(pub_area)   # initial window half-width from published area

    for attempt in range(1, max_tries + 1):
        dlat, dlon = km_to_deg(half_km, lat)
        region = ee.Geometry.Rectangle([lon - dlon, lat - dlat, lon + dlon, lat + dlat])

        dir_tif = f"{WORK}/{name}_dir.tif"
        upa_tif = f"{WORK}/{name}_upa.tif"
        dl(MERIT, "dir", region, 90, dir_tif)
        dl(MERIT, "upa", region, 90, upa_tif)

        grid = Grid.from_raster(dir_tif)
        fdir = grid.read_raster(dir_tif)
        acc  = grid.read_raster(upa_tif)         # upstream drainage area in km2
        aff  = acc.affine
        A    = np.asarray(acc, dtype=float)
        nr, nc = A.shape

        # ---- snap the gauge to the correct channel cell ----
        if name in SNAP_OVERRIDE:
            snap_lon, snap_lat = SNAP_OVERRIDE[name]
            snap_area, snap_dist_km, snap_note = None, 0.0, "manual override"
        else:
            col0, row0 = (~aff) * (lon, lat)
            col0, row0 = int(round(col0)), int(round(row0))
            r_cells = max(3, int(round((5.0 / 111.0) / abs(aff.e))))   # ~5 km search radius
            best = None
            for dr in range(-r_cells, r_cells + 1):
                for dc in range(-r_cells, r_cells + 1):
                    r, c = row0 + dr, col0 + dc
                    if 0 <= r < nr and 0 <= c < nc:
                        a = A[r, c]
                        if not np.isfinite(a) or a <= 0:
                            continue
                        ratio = a / pub_area
                        if 0.3 <= ratio <= 3.0:                        # right size of channel
                            dist = math.hypot(dr, dc)
                            cost = abs(math.log(ratio)) + 0.15 * (dist / r_cells)
                            if best is None or cost < best[0]:
                                best = (cost, r, c, a, dist)
            if best is not None:
                _, r, c, a, dist = best
                snap_lon, snap_lat = aff * (c + 0.5, r + 0.5)
                snap_area = a
                snap_dist_km = dist * abs(aff.e) * 111.0
                snap_note = "area-matched"
            else:
                # fallback: nearest cell in the top 10% accumulation mask
                thr = np.nanpercentile(A[A > 0], 90)
                snap_lon, snap_lat = grid.snap_to_mask(acc > thr, (lon, lat))
                snap_area, snap_dist_km, snap_note = None, None, "fallback high-acc"

        # ---- trace catchment ----
        catch = grid.catchment(x=snap_lon, y=snap_lat, fdir=fdir,
                               dirmap=DIRMAP, xytype="coordinate")
        cb = np.asarray(catch).astype(bool)
        edge_touch = bool(cb[0, :].any() or cb[-1, :].any() or cb[:, 0].any() or cb[:, -1].any())

        # if the basin runs off the window edge, grow the window and retry
        if edge_touch and attempt < max_tries:
            half_km *= 1.7
            print(f"   {name}: catchment hit window edge, enlarging and retrying "
                  f"(attempt {attempt+1})")
            continue

        grid.clip_to(catch)
        shapes = grid.polygonize()
        polys = [shape(g) for g, v in shapes if v]
        poly = largest_polygon(polys)
        area_km2 = geodesic_area_km2(poly)
        return dict(poly=poly, area_km2=area_km2, snap_lon=snap_lon, snap_lat=snap_lat,
                    snap_area=snap_area, snap_dist_km=snap_dist_km, snap_note=snap_note,
                    edge_touch=edge_touch, region=region)

    # last resort return
    grid.clip_to(catch)
    polys = [shape(g) for g, v in grid.polygonize() if v]
    poly = largest_polygon(polys)
    return dict(poly=poly, area_km2=geodesic_area_km2(poly), snap_lon=snap_lon,
                snap_lat=snap_lat, snap_area=snap_area, snap_dist_km=snap_dist_km,
                snap_note=snap_note, edge_touch=True, region=region)

In [32]:
# 6. DEM crop for a catchment polygon
def crop_dem(name, poly):
    minx, miny, maxx, maxy = poly.bounds
    pad = 0.01
    dem_region = ee.Geometry.Rectangle([minx - pad, miny - pad, maxx + pad, maxy + pad])
    dem_full = f"{WORK}/{name}_dem_full.tif"
    dl(GLO30, "DEM", dem_region, 30, dem_full)

    with rasterio.open(dem_full) as src:
        out_img, out_tr = rio_mask(src, [mapping(poly)], crop=True)
        meta = src.meta.copy()
    meta.update(height=out_img.shape[1], width=out_img.shape[2],
                transform=out_tr, compress="deflate")
    dem_out = f"{OUT_DIR}/{name}_DEM_GLO30.tif"
    with rasterio.open(dem_out, "w", **meta) as dst:
        dst.write(out_img)
    return dem_out

In [33]:
# === Cell 7-QA: coordinate on-water / channel check ===
import pandas as pd
JRC = ee.Image("JRC/GSW1_4/GlobalSurfaceWater").select("occurrence")

def coord_qa(lon, lat):
    pt = ee.Geometry.Point([lon, lat])
    occ = JRC.reduceRegion(ee.Reducer.max(), pt.buffer(90), 30).get("occurrence")
    upa = MERIT.select("upa").reduceRegion(ee.Reducer.max(), pt.buffer(200), 90).get("upa")
    return ee.Dictionary({"occ": occ, "upa": upa})

rows = []
for name, lat, lon, pub, basin in STATIONS:
    try:
        d = coord_qa(lon, lat).getInfo()
        occ = None if d.get("occ") is None else round(d["occ"], 0)
        upa = None if d.get("upa") is None else round(d["upa"], 1)
        ratio = None if (upa is None or pub == 0) else round(upa / pub, 2)
        on_water = ((occ or 0) >= 25) or (ratio is not None and 0.3 <= ratio <= 3.0)
        flag = "on-channel" if on_water else "CHECK-off-channel"
        rows.append(dict(station=name, basin=basin, pub_area_km2=pub,
                         water_occ_pct=occ, upa_near_km2=upa, upa_ratio=ratio, coord_flag=flag))
        print(f"{name:16s} occ={occ}  upa_near={upa} km2  ratio={ratio}  {flag}")
    except Exception as e:
        print(f"{name}: QA failed {e}")
        rows.append(dict(station=name, basin=basin, pub_area_km2=pub,
                         water_occ_pct=None, upa_near_km2=None, upa_ratio=None, coord_flag="ERROR"))

qa = pd.DataFrame(rows)
qa.to_csv(f"{OUT_DIR}/coordinate_qa.csv", index=False)

# map: channels + surface water + colored gauge points
Map = geemap.Map(center=[7.7, 80.7], zoom=7)
Map.add_basemap("HYBRID")
Map.addLayer(JRC.updateMask(JRC.gt(10)), {"min": 0, "max": 100,
             "palette": ["cyan", "blue", "navy"]}, "Surface water %")
ch = MERIT.select("upa")
Map.addLayer(ch.updateMask(ch.gt(5)), {"min": 5, "max": 500,
             "palette": ["yellow", "orange", "red"]}, "Channels (upa>5 km2)")
good = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lon, lat]), {"name": name})
        for (name, lat, lon, pub, basin), r in zip(STATIONS, rows) if r["coord_flag"] == "on-channel"])
bad = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lon, lat]), {"name": name})
        for (name, lat, lon, pub, basin), r in zip(STATIONS, rows) if r["coord_flag"] != "on-channel"])
Map.addLayer(good.style(color="00FF00", pointSize=6), {}, "coord on-channel")
Map.addLayer(bad.style(color="FF0000", pointSize=8), {}, "coord CHECK")
display(qa)
Map

Norwood          occ=28  upa_near=98.1 km2  ratio=1.01  on-channel
Kithulgala       occ=None  upa_near=0.1 km2  ratio=0.0  CHECK-off-channel
Deraniyagala     occ=None  upa_near=180.0 km2  ratio=0.98  on-channel
Holombuwa        occ=None  upa_near=0.0 km2  ratio=0.0  CHECK-off-channel
Glencourse       occ=70  upa_near=1524.0 km2  ratio=1.04  on-channel
Hanwella         occ=None  upa_near=1824.5 km2  ratio=1.02  on-channel
Rathnapura       occ=None  upa_near=0.1 km2  ratio=0.0  CHECK-off-channel
Ellagawa         occ=None  upa_near=0.1 km2  ratio=0.0  CHECK-off-channel
Millakanda       occ=None  upa_near=0.0 km2  ratio=0.0  CHECK-off-channel
Putupaula        occ=84  upa_near=2619.6 km2  ratio=1.01  on-channel
Urawa            occ=None  upa_near=76.4 km2  ratio=1.29  on-channel
Pitabeddara      occ=None  upa_near=288.1 km2  ratio=0.93  on-channel
Wellawaya        occ=None  upa_near=15.7 km2  ratio=0.09  CHECK-off-channel
Tanamalwila      occ=None  upa_near=697.2 km2  ratio=0.93  on-channel

,station,basin,pub_area_km2,water_occ_pct,upa_near_km2,upa_ratio,coord_flag
0,Norwood,Kelani Ganga,97,28.0,98.1,1.01,on-channel
1,Kithulgala,Kelani Ganga,383,NaN,0.1,0.00,CHECK-off-channel
2,Deraniyagala,Kelani Ganga,183,NaN,180.0,0.98,on-channel
3,Holombuwa,Kelani Ganga,155,NaN,0.0,0.00,CHECK-off-channel
4,Glencourse,Kelani Ganga,1463,70.0,1524.0,1.04,on-channel
5,Hanwella,Kelani Ganga,1782,NaN,1824.5,1.02,on-channel
6,Rathnapura,Kalu Ganga,603,NaN,0.1,0.00,CHECK-off-channel
7,Ellagawa,Kalu Ganga,1393,NaN,0.1,0.00,CHECK-off-channel
8,Millakanda,Kalu Ganga,780,NaN,0.0,0.00,CHECK-off-channel
9,Putupaula,Kalu Ganga,2598,84.0,2619.6,1.01,on-channel


Map(center=[7.7, 80.7], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…

In [34]:
# === Cell 7d: 30 m catchment delineation (Copernicus GLO-30 + pysheds) ===
import pandas as pd
from shapely.geometry import mapping
TOL = 0.10

COORD_OVERRIDE = {}   # replace a gauge coord flagged off-channel in QA, e.g. "Nakkala": (81.2960, 6.8955)
SNAP_OVERRIDE  = {}   # force the pour cell exactly, e.g. "Peradeniya": (80.6090, 7.2670)

def delineate_30m(name, lat, lon, pub_area, basin, max_tries=3):
    if name in COORD_OVERRIDE:
        lon, lat = COORD_OVERRIDE[name]
    half_km = 12 + 1.0 * math.sqrt(pub_area)
    for attempt in range(1, max_tries + 1):
        dlat, dlon = km_to_deg(half_km, lat)
        region = ee.Geometry.Rectangle([lon - dlon, lat - dlat, lon + dlon, lat + dlat])
        dem_path = f"{WORK}/{name}_glo30.tif"
        if os.path.exists(dem_path):
            os.remove(dem_path)
        geemap.download_ee_image(GLO30, dem_path, region=region, scale=30, crs="EPSG:4326")

        grid = Grid.from_raster(dem_path)
        dem = grid.read_raster(dem_path)
        pit      = grid.fill_pits(dem)
        flooded  = grid.fill_depressions(pit)
        inflated = grid.resolve_flats(flooded)
        fdir = grid.flowdir(inflated, dirmap=DIRMAP)
        acc  = grid.accumulation(fdir, dirmap=DIRMAP)   # cell counts, same grid as fdir

        T = acc.affine
        A = np.asarray(acc, dtype="float64")
        H, Wd = A.shape
        dy_m = abs(T.e) * 111000.0
        dx_m = abs(T.a) * 111000.0 * math.cos(math.radians(lat))
        cell_km2 = (dy_m * dx_m) / 1e6
        target_cells = pub_area / cell_km2

        if name in SNAP_OVERRIDE:
            plon, plat = SNAP_OVERRIDE[name]
            sdist_m, snap_acc_km2 = 0.0, None
        else:
            col0, row0 = (~T) * (lon, lat)
            row0, col0 = int(round(row0)), int(round(col0))
            rc = max(5, int(round((6.0 / 111.0) / abs(T.e))))   # ~6 km search radius
            best = None
            for dr in range(-rc, rc + 1):
                for dc in range(-rc, rc + 1):
                    r, c = row0 + dr, col0 + dc
                    if 0 <= r < H and 0 <= c < Wd:
                        a = A[r, c]
                        if a <= 0 or not np.isfinite(a):
                            continue
                        ratio = a / target_cells
                        if 0.25 <= ratio <= 4.0:
                            dm = math.hypot(dr * dy_m, dc * dx_m)
                            cost = abs(math.log(ratio)) + 0.08 * (dm / (rc * dy_m))
                            if best is None or cost < best[0]:
                                best = (cost, r, c, a, dm)
            if best is None:
                rr, cc = np.unravel_index(np.nanargmax(A), A.shape)
                best = (0, rr, cc, A[rr, cc], math.hypot((rr - row0) * dy_m, (cc - col0) * dx_m))
            _, r, c, a, dm = best
            plon, plat = T * (c + 0.5, r + 0.5)
            sdist_m, snap_acc_km2 = dm, a * cell_km2

        catch = grid.catchment(x=plon, y=plat, fdir=fdir, dirmap=DIRMAP,
                               xytype="coordinate", snap="center")
        cb = np.asarray(catch).astype(bool)
        edge = bool(cb[0, :].any() or cb[-1, :].any() or cb[:, 0].any() or cb[:, -1].any())
        if edge and attempt < max_tries:
            half_km *= 1.7
            continue

        grid.clip_to(catch)
        polys = [shape(g) for g, v in grid.polygonize() if v]
        poly = largest_polygon(polys)
        area = geodesic_area_km2(poly)

        # crop the 30 m DEM we already have to the catchment
        with rasterio.open(dem_path) as src:
            out_img, out_tr = rio_mask(src, [mapping(poly)], crop=True)
            meta = src.meta.copy()
        meta.update(height=out_img.shape[1], width=out_img.shape[2],
                    transform=out_tr, compress="deflate")
        with rasterio.open(f"{OUT_DIR}/{name}_DEM_GLO30.tif", "w", **meta) as dst:
            dst.write(out_img)

        return dict(poly=poly, area=area, plon=plon, plat=plat,
                    sdist_m=sdist_m, snap_acc_km2=snap_acc_km2, edge=edge)
    return None

summary, merged = [], []
for name, lat, lon, pub_area, basin in STATIONS:
    print(f"-> {name} ({basin}, published {pub_area} km2)")
    try:
        res = delineate_30m(name, lat, lon, pub_area, basin)
        if res is None:
            raise RuntimeError("could not delineate")
        poly, area = res["poly"], res["area"]
        gdf = gpd.GeoDataFrame(
            dict(station=[name], basin=[basin], pub_area_km2=[pub_area],
                 del_area_km2=[round(area, 1)],
                 snap_acc_km2=[None if res["snap_acc_km2"] is None else round(res["snap_acc_km2"], 1)],
                 snap_dist_m=[round(res["sdist_m"], 0)],
                 snap_lon=[round(res["plon"], 5)], snap_lat=[round(res["plat"], 5)]),
            geometry=[poly], crs="EPSG:4326")
        gdf.to_file(f"{OUT_DIR}/{name}_catchment.geojson", driver="GeoJSON")
        gdf.to_file(f"{OUT_DIR}/{name}_catchment.shp")
        merged.append(gdf)
        pct = 100.0 * (area - pub_area) / pub_area
        flag = "OK" if abs(pct) <= 100 * TOL else "CHECK"
        if res["edge"]:
            flag = "CHECK(edge)"
        summary.append(dict(station=name, basin=basin, pub_area_km2=pub_area,
                            del_area_km2=round(area, 1), diff_pct=round(pct, 1),
                            snap_dist_m=round(res["sdist_m"], 0), flag=flag))
        print(f"   {area:.1f} km2 ({pct:+.1f}%)  snap {res['sdist_m']:.0f} m  [{flag}]")
    except Exception as e:
        print("   FAILED:", e)
        summary.append(dict(station=name, basin=basin, pub_area_km2=pub_area,
                            del_area_km2=None, diff_pct=None, snap_dist_m=None, flag="FAILED"))

sdf = pd.DataFrame(summary)
sdf.to_csv(f"{OUT_DIR}/delineation_summary.csv", index=False)
if merged:
    all_gdf = gpd.GeoDataFrame(pd.concat(merged, ignore_index=True), crs="EPSG:4326")
    all_gdf.to_file(f"{OUT_DIR}/all_catchments.geojson", driver="GeoJSON")
print("\nDone. Cross-check snap_dist_m against the QA table for anything flagged.")
sdf

-> Norwood (Kelani Ganga, published 97 km2)


  0%|          |0/3 tiles [00:00<?]

   97.4 km2 (+0.4%)  snap 84 m  [OK]
-> Kithulgala (Kelani Ganga, published 383 km2)


  0%|          |0/10 tiles [00:00<?]

   390.3 km2 (+1.9%)  snap 1410 m  [OK]
-> Deraniyagala (Kelani Ganga, published 183 km2)


...

  0%|          |0/14 tiles [00:00<?]

   960.5 km2 (-29.4%)  snap 1349 m  [CHECK]
-> Dunamale (Aththanagalu Oya, published 153 km2)


  0%|          |0/4 tiles [00:00<?]

   150.1 km2 (-1.9%)  snap 348 m  [OK]

Done. Cross-check snap_dist_m against the QA table for anything flagged.


,station,basin,pub_area_km2,del_area_km2,diff_pct,snap_dist_m,flag
0,Norwood,Kelani Ganga,97,97.4,0.4,84.0,OK
1,Kithulgala,Kelani Ganga,383,390.3,1.9,1410.0,OK
2,Deraniyagala,Kelani Ganga,183,179.8,-1.8,191.0,OK
3,Holombuwa,Kelani Ganga,155,159.0,2.6,320.0,OK
4,Glencourse,Kelani Ganga,1463,1465.4,0.2,1064.0,OK
5,Hanwella,Kelani Ganga,1782,1726.3,-3.1,1783.0,OK
6,Rathnapura,Kalu Ganga,603,630.3,4.5,565.0,OK
7,Ellagawa,Kalu Ganga,1393,1368.6,-1.8,209.0,OK
8,Millakanda,Kalu Ganga,780,615.5,-21.1,4955.0,CHECK
9,Putupaula,Kalu Ganga,2598,2418.9,-6.9,2038.0,OK


In [36]:
# 8. Quick visual QA map (catchments over satellite imagery)
import folium, geopandas as gpd

all_gdf = gpd.read_file(f"{OUT_DIR}/all_catchments.geojson")

m = folium.Map(location=[7.7, 80.7], zoom_start=7, tiles=None)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Esri Satellite").add_to(m)

# color catchments by flag from the summary
flags = dict(zip(sdf.station, sdf.flag))
def style(feat):
    f = flags.get(feat["properties"]["station"], "OK")
    col = "#e34a33" if str(f).startswith("CHECK") else "#00e5ff"
    return {"color": col, "weight": 1.8, "fillColor": col, "fillOpacity": 0.12}

folium.GeoJson(all_gdf, name="Catchments", style_function=style,
               tooltip=folium.GeoJsonTooltip(fields=["station", "del_area_km2", "pub_area_km2"])
               ).add_to(m)

# gauge points (published coordinate) + snapped pour points
for name, lat, lon, pub, basin in STATIONS:
    folium.CircleMarker([lat, lon], radius=3, color="yellow", fill=True,
                        fill_opacity=1, popup=f"{name} (gauge)").add_to(m)
for _, r in all_gdf.iterrows():
    folium.CircleMarker([r.snap_lat, r.snap_lon], radius=3, color="lime", fill=True,
                        fill_opacity=1, popup=f"{r.station} (snapped)").add_to(m)

folium.LayerControl().add_to(m)
m

## Notes for the next steps

- **Validation.** The `flag` column flags any catchment where the delineated area
  differs from the published area by more than 15%, or where the basin touched the
  download window (auto-grown, but worth a look). The methodology expects 20 of 22
  within 10%; a couple near stream junctions may need a manual pour point.
- **Fixing a bad snap.** If a gauge snapped to the wrong tributary, put its correct
  channel coordinate in `SNAP_OVERRIDE` (cell 3) as `(lon, lat)` and re-run cells 5 to 7.
  You can read a good pour point straight off the QA map in cell 8.
- **Outputs in Drive** (`AGBD_Hydrology/01_catchments/`): per-station
  `<Name>_catchment.geojson`, `<Name>_catchment.shp` (+ sidecars),
  `<Name>_DEM_GLO30.tif`, plus `all_catchments.geojson` and `delineation_summary.csv`.
- **Step 2** (zonal AGBD, slope, elevation, soil, land use per catchment) reuses
  `all_catchments.geojson` directly in Earth Engine.